<a href="https://colab.research.google.com/github/danguibes/monitor-precos/blob/main/Oxylabs_Teste.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
TOKEN = "1gl1zRFadZQzSuJZnWmZeA0hsVNTuuqO41mn3a42h89T"

In [7]:
import os
import json
import time
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

API_URL = "https://webapi.oxylabs.io/v1/scrape"
# TOKEN = os.environ["OXYLABS_TOKEN"]  # export OXYLABS_TOKEN="seu_token"
HEADERS = {"Content-Type": "application/json", "Authorization": f"Bearer {TOKEN}"}

#BASE_URL = "https://www.magazineluiza.com.br/sofas/moveis/s/mo/msof/"

BASE_URL = "https://lista.mercadolivre.com.br/sofa#D[A:sofa]"
N_PAGES = 1
LINKS_CSV = "magalu_sofas_links.csv"
DETAILS_CSV = "magalu_sofas_detalhes.csv"
DETAILS_JSONL = "magalu_sofas_detalhes_raw.jsonl"  # backup bruto, permite retomar
MAX_WORKERS = 5

PROMPT = """
Extract the following information from this product page and return a flat JSON object
with exactly these keys (use null when a value is not available on the page):
- product_name
- brand
- seller
- price_cash: the full "à vista" price (number, BRL)
- price_pix: the price when paying with Pix (number, BRL)
- price_installment_total: the total price when paying in installments (number, BRL)
- installments_count: the maximum number of installments (integer)
- installment_value: the value of each installment (number, BRL)
- interest_free: whether the installments are interest-free (true/false)
- color
- material: upholstery/fabric/structure material
- dimensions: object with width_cm, depth_cm, height_cm (numbers) and any other dimension reported
- weight_kg
- seats: number of seats
- reclinable: true/false
- retractable: true/false
- sofa_bed: true/false
- has_usb: true/false
- has_cup_holder: true/false
- doors_count: number of doors/drawers/compartments, if any
- warranty
- description: short product description
- specifications: object with ALL technical characteristics listed on the page
  as key/value pairs (keep original keys, e.g. composition, filling, feet type, load capacity,
  box included, assembly required, etc.)
Return only the JSON.
""".strip()


def oxylabs(payload, retries=3):
    for attempt in range(1, retries + 1):
        try:
            r = requests.post(API_URL, json=payload, headers=HEADERS, timeout=180)
            if r.ok:
                return r.json()
            print(f"  HTTP {r.status_code}: {r.text[:800]}")
            if r.status_code in (400, 401, 403):
                return None  # erro de requisição/credencial: repetir não adianta
        except Exception as e:
            print(f"  erro: {e}")
        time.sleep(2 * attempt)
    return None


# ---------------------------------------------------------------- FASE 1
LISTING_PROMPT = (
    'Return a JSON object {"products": [...]} listing every product shown on this listing page. '
    "Each item must have: name, price (number, BRL) and url (the full absolute product link, exactly as in the page)."
)


def extract_products(js):
    """Aceita {'products': [...]} ou qualquer primeira lista de dicts encontrada."""
    if isinstance(js, list):
        return [p for p in js if isinstance(p, dict)]
    if isinstance(js, dict):
        if isinstance(js.get("products"), list):
            return js["products"]
        for v in js.values():
            if isinstance(v, list) and v and isinstance(v[0], dict):
                return v
    return []


def phase1_collect_links():
    rows = []
    for page in range(1, N_PAGES + 1):
        url = BASE_URL if page == 1 else f"{BASE_URL}?page={page}"
        print(f"[fase 1] página {page}: {url}")
        # output "json" exige json.prompt ou json.schema; usamos um prompt mínimo só p/ listar produtos
        data = oxylabs(
            {
                "url": url,
                "output": ["json"],
                "json": {"prompt": LISTING_PROMPT},
                "run_js": False,
            }
        )
        if not data:
            print("  falhou, pulando")
            continue
        products = extract_products(data["results"][0]["json"])
        print(f"  {len(products)} produtos")
        for p in products:
            rows.append(
                {
                    "page": page,
                    "name": p.get("name"),
                    "listing_price": p.get("price"),
                    # as URLs vêm da versão mobile; normaliza para desktop
                    "url": (p.get("url") or "").replace("://m.magazineluiza", "://www.magazineluiza"),
                }
            )
    df = pd.DataFrame(rows).drop_duplicates(subset="url")
    df.to_csv(LINKS_CSV, index=False, encoding="utf-8-sig")
    print(f"[fase 1] {len(df)} links únicos salvos em {LINKS_CSV}")
    return df


# ---------------------------------------------------------------- FASE 2
def scrape_product(url):
    data = oxylabs({"url": url, "output": ["json"], "json": {"prompt": PROMPT}, "run_js": False})
    if not data:
        return {"url": url, "error": True}
    return {"url": url, "result": data["results"][0].get("json")}


def flatten(url, result):
    row = {"url": url}
    if not isinstance(result, dict):
        row["raw"] = json.dumps(result, ensure_ascii=False)
        return row
    for k, v in result.items():
        if k == "specifications" and isinstance(v, dict):
            for sk, sv in v.items():
                row[f"spec_{sk}"] = sv
        elif k == "dimensions" and isinstance(v, dict):
            for dk, dv in v.items():
                row[f"dim_{dk}"] = dv
        elif isinstance(v, (dict, list)):
            row[k] = json.dumps(v, ensure_ascii=False)
        else:
            row[k] = v
    return row


def phase2_scrape_details(urls):
    done = set()
    if os.path.exists(DETAILS_JSONL):
        with open(DETAILS_JSONL, encoding="utf-8") as f:
            for line in f:
                done.add(json.loads(line)["url"])
    todo = [u for u in urls if u not in done]
    print(f"[fase 2] {len(done)} já feitos, {len(todo)} a raspar")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex, open(DETAILS_JSONL, "a", encoding="utf-8") as out:
        futures = {ex.submit(scrape_product, u): u for u in todo}
        for i, fut in enumerate(as_completed(futures), 1):
            res = fut.result()
            if res.get("error"):
                print(f"  [{i}/{len(todo)}] ERRO {res['url']}")
                continue
            out.write(json.dumps(res, ensure_ascii=False) + "\n")
            out.flush()
            print(f"  [{i}/{len(todo)}] ok")

    rows = []
    with open(DETAILS_JSONL, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            rows.append(flatten(r["url"], r["result"]))
    df = pd.DataFrame(rows)
    df.to_csv(DETAILS_CSV, index=False, encoding="utf-8-sig")
    print(f"[fase 2] {len(df)} produtos salvos em {DETAILS_CSV}")
    return df


if __name__ == "__main__":
    if os.path.exists(LINKS_CSV):
        links = pd.read_csv(LINKS_CSV)
        print(f"Usando {LINKS_CSV} existente ({len(links)} links). Apague o arquivo para refazer a fase 1.")
    else:
        links = phase1_collect_links()
    phase2_scrape_details(links["url"].tolist())

[fase 1] página 1: https://lista.mercadolivre.com.br/sofa#D[A:sofa]
  HTTP 500: {"status":500,"title":"REQUEST_FAILED_AFTER_TOO_MANY_RETRIES","detail":"The request failed after the maximum number of retries. You can try again at no extra cost, as we don't charge you for faulted scrapes.","instance":"6af63b54-39103f8ee2ddff3efa7825c3","state":"faulted","params":{"url":"https://lista.mercadolivre.com.br/sofa#D[A:sofa]","output":["json"],"json":{"schema":null,"prompt":"Return a JSON object {\"products\": [...]} listing every product shown on this listing page. Each item must have: name, price (number, BRL) and url (the full absolute product link, exactly as in the page)."},"location":null,"device":"desktop","run_js":false},"metadata":{"timestamp":1794521940,"request_id":"7526770553827393537"}}

  HTTP 500: {"status":500,"title":"REQUEST_FAILED_AFTER_TOO_MANY_RETRIES","detail":"The request failed after the maximum number of retries. You can try again at no extra cost, as we don't charge yo

In [ ]:
import re
import json
import pandas as pd

# usa `url` (endpoint), `headers` e `requests` já definidos na célula anterior
link = pd.read_csv("magalu_sofas_links.csv")["url"].iloc[0]
print("URL:", link)

r = requests.post(url, json={"url": link, "output": ["html"], "run_js": False}, headers=headers, timeout=180)
print("HTTP", r.status_code)
if not r.ok:
    print(r.text[:800])
    raise SystemExit
resp = r.json()
result = resp["results"][0]

print("chaves do resultado:", list(result.keys()))
print("metadata:", json.dumps(result.get("metadata"), ensure_ascii=False)[:500])

# tenta os nomes de campo mais prováveis
html = result.get("html") or result.get("content") or ""
if isinstance(html, dict):  # caso venha aninhado
    html = html.get("html") or html.get("content") or json.dumps(html, ensure_ascii=False)
print("tamanho do HTML:", len(html))

if not html:
    # salva a resposta inteira para eu conseguir ver o formato
    with open("resposta_completa.json", "w", encoding="utf-8") as f:
        json.dump(resp, f, ensure_ascii=False, indent=2)
    print("HTML vazio: salvei resposta_completa.json — me mande esse arquivo")
else:
    with open("produto_exemplo.html", "w", encoding="utf-8") as f:
        f.write(html)
    ld = re.findall(r'<script[^>]*type="application/ld\+json"[^>]*>(.*?)</script>', html, flags=re.S)
    m = re.search(r'<script[^>]*id="__NEXT_DATA__"[^>]*>(.*?)</script>', html, flags=re.S)
    with open("trechos_estruturados.json", "w", encoding="utf-8") as f:
        json.dump({"ld_json": ld, "next_data": m.group(1) if m else None}, f, ensure_ascii=False, indent=2)
    print(f"ld+json: {len(ld)} | __NEXT_DATA__: {'sim' if m else 'não'}")

URL: https://www.magazineluiza.com.br/sofa-3-lugares-2m-material-sintetico-retro-elegante-recepcoes-escritorios-sala-estar-living-classic-ms-decor/p/dd5j4719dk/mo/sa03/?seller_id=leiiloesdosul
HTTP 200
chaves do resultado: ['html', 'metadata']
metadata: {"page": 1, "run_js": false, "url": "https://m.magazineluiza.com.br/sofa-3-lugares-2m-material-sintetico-retro-elegante-recepcoes-escritorios-sala-estar-living-classic-ms-decor/p/dd5j4719dk/mo/sa03/?seller_id=leiiloesdosul", "statuses": {"http_request": {"code": 200, "tag": "HTTP_OK"}}}
tamanho do HTML: 830556
ld+json: 3 | __NEXT_DATA__: sim


In [ ]:
import os
import re
import json
import html as htmllib
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# Reaproveita da célula anterior do notebook: `requests`, `url` (endpoint) e `headers` (token)
LINKS_CSV = "magalu_sofas_links.csv"
RAW_JSONL = "magalu_sofas_next_data.jsonl"   # __NEXT_DATA__ bruto por produto (permite reparsear sem gastar)
OUT_CSV = "magalu_sofas_detalhes_local.csv"   # (o magalu_sofas_detalhes.csv da IA fica intacto)
MAX_WORKERS = 5
LIMIT = 25          # ex.: 5 para testar com poucos links
SAVE_HTML = False     # True salva o HTML completo em html/<id>.html (~800 KB cada)

NEXT_RE = re.compile(r'<script[^>]*id="__NEXT_DATA__"[^>]*>(.*?)</script>', re.S)


# ------------------------------------------------------------------ download
def fetch_next_data(link, retries=3):
    import time
    for attempt in range(1, retries + 1):
        try:
            r = requests.post(
                url, json={"url": link, "output": ["html"], "run_js": False}, headers=headers, timeout=180
            )
            if r.ok:
                page = r.json()["results"][0].get("html") or ""
                if SAVE_HTML and page:
                    os.makedirs("html", exist_ok=True)
                    pid = re.search(r"/p/([a-z0-9]+)/", link)
                    with open(f"html/{pid.group(1) if pid else abs(hash(link))}.html", "w", encoding="utf-8") as f:
                        f.write(page)
                m = NEXT_RE.search(page)
                if m:
                    return {"link": link, "next_data": json.loads(m.group(1))}
                print(f"  sem __NEXT_DATA__: {link[:90]}")
            else:
                print(f"  HTTP {r.status_code}: {r.text[:200]}")
                if r.status_code in (400, 401, 403):
                    break
        except Exception as e:
            print(f"  erro: {e}")
        time.sleep(2 * attempt)
    return {"link": link, "error": True}


# ------------------------------------------------------------------ normalizadores (regex)
NUM = r"\d+(?:[.,]\d+)?"
UNIT = r"(?:cm|m)\b"


def _f(x):
    return float(str(x).replace(",", "."))


def to_cm(v, unit=None):
    """Valor -> cm. 'm' => *100; sem unidade ou 'cm' com valor < 10 => era metro mal rotulado ('2.75cm', '0,92')."""
    v = _f(v)
    u = (unit or "").lower()
    cm = v * 100 if (u == "m" or v < 10) else v
    return round(cm, 1) if 20 <= cm <= 700 else None   # descarta pés, braços, lixo


# ----------------------------------------------------------------------------- lugares / configurações
def seats_from(*texts):
    """Maior nº de lugares citado ('3 lugares', '2 e 3 lugares', '03', '5')."""
    found = []
    for t in texts:
        t = str(t or "").lower()
        for m in re.finditer(r"\b(\d{1,2})(?:\s*(?:e|ou|/)\s*(\d{1,2}))?\s*lugares?\b", t):
            found += [int(g) for g in m.groups() if g]
        if re.fullmatch(r"\s*\d{1,2}\s*", t):
            found.append(int(t))
    found = [n for n in found if 1 <= n <= 9]
    return max(found) if found else None


SPLIT = re.compile(
    r"(?=\b\d\s*lugares?\s*[:\-–]|\bpe[çc]a de\b|\bsof[áa]\s*(?:maior|menor)\s*:|\bsof[áa](?:-cama)?\s*-\s|\bsentado\b|\bdeitado\b)",
    re.I,
)


def _seg_seats(p):
    m = re.search(r"(\d)\s*lugares?", p[:30], re.I)
    return int(m.group(1)) if m else None


def pick_segment(text, seats=None):
    """Quebra 'configurações' (2 lugares / 3 lugares, peça de.., sentado/deitado) e escolhe a do nº de lugares."""
    parts = [p.strip(" ;.|,") for p in SPLIT.split(str(text)) if p and re.search(r"\d", p)]
    parts = [p.strip(" ;.|,") for p in parts]
    if len(parts) <= 1:
        return (parts[0] if parts else str(text)), False
    if seats:
        for p in parts:
            if _seg_seats(p) == seats:
                return p, True
    with_seats = [(p, _seg_seats(p)) for p in parts if _seg_seats(p)]
    if with_seats:                       # conjunto 2+3 lugares: usa a maior configuração
        return max(with_seats, key=lambda x: x[1])[0], True
    return parts[0], True


# ----------------------------------------------------------------------------- peso
KEY_W_PRODUTO = [
    r"peso (aproximado )?-? ?(peso )?(do )?produto( \(kg\))?",
    r"peso l[ií]quido( do produto)?( \(kg\))?",
    r"peso do item.*", r"peso \(kg\)", r"peso", r"peso do sof[áa]( \(kg\))?",
]
KEY_W_EMB = [r"peso.*embalagem.*", r"peso bruto.*"]


def parse_weight_kg(text, seats=None):
    """'62 kg', '61,20', '3 lugares: 56,6 kg. 2 lugares: 49,1 kg' -> kg (usa a configuração do nº de lugares)."""
    if text in (None, ""):
        return None
    seg, _ = pick_segment(str(text), seats)
    seg = re.sub(r"^(?:pe[çc]a de\s+|sof[áa]\s+)?\d\s*lugares?\s*[:\-–]?\s*", "", seg.strip(), flags=re.I)
    m = re.search(rf"({NUM})\s*(kg|g)?\b", seg.lower())
    if not m:
        return None
    v = _f(m.group(1)) / (1000 if m.group(2) == "g" else 1)
    return v if 3 <= v <= 400 else None


def weight_from_ficha(ficha, seats=None):
    """Devolve (peso_produto, peso_embalagem) escolhendo a 1ª chave existente, em ordem de prioridade."""
    def find(patterns, exclude=None):
        for pat in patterns:
            for k, v in ficha.items():
                kk = k.lower().strip()
                if re.fullmatch(pat, kk) and not (exclude and re.search(exclude, kk)):
                    w = parse_weight_kg(v, seats)
                    if w:
                        return w
        return None
    emb = find(KEY_W_EMB)
    prod = find(KEY_W_PRODUTO, exclude=r"embala|suport|bruto")
    return prod, emb


def weight_from_text(desc):
    m = re.search(rf"peso(?: l[ií]quido| do produto| aproximado)?\s*[:\-]?\s*({NUM})\s*kg", str(desc or ""), re.I)
    return parse_weight_kg(m.group(1) + " kg") if m else None


# ----------------------------------------------------------------------------- espuma
def parse_foam(*texts):
    found = []
    for t in texts:
        for m in re.finditer(r"\bD[- ]?(\d{2})\b", str(t or ""), flags=re.I):
            code = f"D{m.group(1)}"
            if code not in found:
                found.append(code)
    return " / ".join(found) if found else None


# ----------------------------------------------------------------------------- medidas
LABEL = {"largura": "W", "comprimento": "W", "profundidade": "D", "altura": "H"}
STATE = {"aberto": "aberto", "aberta": "aberto", "deitado": "aberto", "fechado": "fechado", "fechada": "fechado",
         "sentado": "fechado", "total": None, "totalmente": None}
IGNORE = {"braço", "braços", "braco", "bracos", "cada", "encosto", "encostos", "menor", "módulo", "módulos", "modulo",
          "modulos", "chaise", "abertura", "almofada", "almofadas", "pés", "pé", "pes", "pe", "bau", "baú"}
TOK = re.compile(rf"({NUM})(?:\s*({UNIT}))?(\s*°)?|([a-zà-ú]+)", re.I)


def _triplet(seg):
    m = re.search(
        rf"({NUM})\s*({UNIT})?\s*[x×]\s*({NUM})\s*({UNIT})?\s*[x×]\s*({NUM})\s*({UNIT})?", seg, re.I
    )
    if not m:
        return None
    g = m.groups()
    unit = g[1] or g[3] or g[5]
    vals = [to_cm(g[0], g[1] or unit), to_cm(g[2], g[3] or unit), to_cm(g[4], g[5] or unit)]
    if None in vals:
        return None
    _triplet.each_unit = bool(g[1] and g[3])      # "90cm x 1,95m x 82cm" vs "96 x 190 x 88 cm"
    return vals



LAB_NUM = re.compile(rf"(altura|profundidade)\b[^\d\n]{{0,25}}?({NUM})\s*(cm|m)?\b|({NUM})\s*(cm|m)?\s*de\s*(altura|profundidade)", re.I)


def labeled_from_context(context):
    """Valores 'altura'/'profundidade' declarados em texto livre (descrição, outras chaves da ficha)."""
    found = {"altura": set(), "profundidade": set()}
    for m in LAB_NUM.finditer(str(context or "")):
        if m.group(1):
            lab, num, unit = m.group(1), m.group(2), m.group(3)
        else:
            lab, num, unit = m.group(6), m.group(4), m.group(5)
        v = to_cm(num, unit)
        if v:
            found[lab.lower()].add(v)
    return found

POST = re.compile(
    rf"({NUM})\s*({UNIT})?\s*(?:\(\s*|de\s+|do\s+)?(largura|comprimento|profundidade|altura)\b(?:\s*\)?\s*(aberto|fechado|total)\b)?",
    re.I,
)
PRE_COUNT = re.compile(rf"(largura|comprimento|profundidade|altura)\b[^\d:]{{0,25}}[:\-]?\s*{NUM}", re.I)
HINT = re.compile(r"\(\s*([alp])\s*[x×]\s*([alp])\s*[x×]\s*([alp])\s*\)", re.I)


def _pre_mode(seg):
    got = {}
    label = state = None
    ignore = False
    prev = ""
    for m in TOK.finditer(seg):
        num, unit, deg, word = m.groups()
        if word:
            w = word.lower()
            if w in LABEL:
                label, state, ignore = LABEL[w], None, False
            elif w in STATE:
                state = STATE[w]
            elif w in IGNORE or (w == "assento" and prev in {"do", "da", "dos", "das"}):
                ignore = True
            prev = w
            continue
        if deg or label is None:
            prev = ""
            continue
        if not ignore:
            v = to_cm(num, unit)
            if v is not None:
                got.setdefault((label, state or "base"), v)
        state = None
        prev = ""
    return got


def _post_mode(seg):
    got = {}
    for m in POST.finditer(seg):
        num, unit, lab, st = m.groups()
        tail = seg[m.end(): m.end() + 14].lower()
        if re.match(r"\s*(do|da|dos)\s+(br|ch|c[ao]d)", tail):   # "de profundidade dos módulos"
            continue
        v = to_cm(num, unit)
        if v is not None:
            got.setdefault((LABEL[lab.lower()], (STATE.get((st or "").lower()) or "base")), v)
    return got


def parse_dimensions(text, seats=None, context=None):
    """
    Retorna: largura_cm, profundidade_cm (fechada/base), altura_cm, profundidade_aberto_cm,
    dim_modo ('rotulado' | 'rotulo_depois' | 'dica_AxLxP' | 'triplo_inferido' | 'valor_unico' | None),
    dim_complexa (várias configurações no texto).
    Triplo sem rótulo: maior = largura; os outros dois na ordem do texto = profundidade, altura (padrão P x L x A
    observado nas fichas). Marcado como 'triplo_inferido' para revisão.
    """
    out = dict(largura_cm=None, profundidade_cm=None, altura_cm=None, profundidade_aberto_cm=None,
               dim_modo=None, dim_complexa=False, altura_aberto_cm=None, dim_ordem_incerta=False)
    if not text:
        return out
    seg, complexa = pick_segment(str(text), seats)
    out["dim_complexa"] = complexa

    # 1) dica explícita da ordem: "95x150x90cm (AxLxP)"
    h = HINT.search(seg)
    t = _triplet(seg)
    if h and t:
        order = [c.upper() for c in h.groups()]
        key = {"A": "altura_cm", "L": "largura_cm", "P": "profundidade_cm"}
        out.update({key[c]: v for c, v in zip(order, t)})
        out["dim_modo"] = "dica_AxLxP"
        return out

    # 2) rótulos (antes ou depois do número, o que aparecer mais)
    if re.search(r"largura|comprimento|profundidade|altura", seg, re.I):
        pre, post = _pre_mode(seg), _post_mode(seg)
        n_pre, n_post = len(PRE_COUNT.findall(seg)), len(re.findall(POST, seg))
        got, modo = (post, "rotulo_depois") if (len(post) >= 2 and n_post > n_pre) else (pre, "rotulado")
        if len(got) < len(post):
            got, modo = post, "rotulo_depois"
        out["largura_cm"] = got.get(("W", "base")) or got.get(("W", "fechado"))
        out["profundidade_cm"] = got.get(("D", "base")) or got.get(("D", "fechado"))
        out["altura_cm"] = got.get(("H", "base")) or got.get(("H", "fechado"))
        out["profundidade_aberto_cm"] = got.get(("D", "aberto"))
        out["altura_aberto_cm"] = got.get(("H", "aberto"))
        if any(out[k] for k in ("largura_cm", "profundidade_cm", "altura_cm")):
            out["dim_modo"] = modo
            return out

    # 3) três números sem rótulo: maior = largura. Para os outros dois (altura x profundidade):
    #    a) um valor > 110 cm é a profundidade (sofá retrátil/aberto);
    #    b) senão, confere com 'altura'/'profundidade' declaradas na descrição/ficha;
    #    c) senão, estilo: unidade em cada número => A x L x P; unidade só no fim => P x L x A (marcado como incerto).
    if t:
        w_i = max(range(3), key=lambda i: t[i])
        a_, b_ = [t[i] for i in range(3) if i != w_i]
        incerta = False
        if a_ > 110 >= b_:
            prof, alt = a_, b_
        elif b_ > 110 >= a_:
            prof, alt = b_, a_
        else:
            ctx = labeled_from_context(context)
            if a_ in ctx["altura"] and b_ not in ctx["altura"] or b_ in ctx["profundidade"] and a_ not in ctx["profundidade"]:
                alt, prof = a_, b_
            elif b_ in ctx["altura"] and a_ not in ctx["altura"] or a_ in ctx["profundidade"] and b_ not in ctx["profundidade"]:
                alt, prof = b_, a_
            else:
                incerta = True
                alt, prof = (a_, b_) if getattr(_triplet, "each_unit", False) else (b_, a_)
        out.update(largura_cm=t[w_i], profundidade_cm=prof, altura_cm=alt, dim_modo="triplo_inferido",
                   dim_ordem_incerta=incerta)
        return out

    # 4) valor único ("1,80m", "200cm") = largura
    m = re.fullmatch(rf"\s*({NUM})\s*({UNIT})?\s*", seg, re.I)
    if m and to_cm(m.group(1), m.group(2)):
        out.update(largura_cm=to_cm(m.group(1), m.group(2)), dim_modo="valor_unico")
    return out


def width_from_title(title):
    m = re.search(rf"\b(\d[.,]\d{{1,2}})\s*m\b", str(title or ""), re.I)
    return to_cm(m.group(1), "m") if m else None


# ------------------------------------------------------------------ parsing
def strip_html(s):
    s = re.sub(r"<br\s*/?>|</p>|</li>", "\n", s or "")
    s = re.sub(r"<[^>]+>", "", s)
    return re.sub(r"\n{2,}", "\n", htmllib.unescape(s)).strip()


def slug(s):
    return re.sub(r"[^a-z0-9]+", "_", re.sub(r"[áàâãä]", "a", re.sub(r"[éèêë]", "e", re.sub(r"[íìîï]", "i",
           re.sub(r"[óòôõö]", "o", re.sub(r"[úùûü]", "u", re.sub(r"ç", "c", s.lower()))))))).strip("_")


def parse_product(link, nd):
    data = nd["props"]["pageProps"]["data"]
    item = data["item"]
    offer = (item.get("offers") or [{}])[0]
    row = {"link": link, "id": item.get("id"), "titulo": item.get("title"), "marca": (item.get("brand") or {}).get("label"),
           "categoria": (item.get("category") or {}).get("name"), "subcategoria": (item.get("subcategory") or {}).get("name"),
           "disponivel": item.get("available")}

    # --- vendedor
    sp = data.get("sellerPage") or {}
    seller = offer.get("seller") or {}
    saddr = seller.get("address") or {}
    row.update({"vendedor": sp.get("name") or seller.get("description"), "vendedor_id": seller.get("id"),
                "vendedor_cidade": saddr.get("city"), "vendedor_uf": saddr.get("state"),
                "entrega_por": seller.get("deliveryDescription")})

    # --- preços
    best_pix = offer.get("bestPrice") or {}
    best_inst = offer.get("bestInstallmentPlan") or {}
    row.update({
        "preco_lista": offer.get("listPrice"),
        "preco_a_vista": offer.get("price"),   # preço base (1x sem desconto)
        "preco_pix": best_pix.get("totalAmount") if best_pix.get("paymentMethodId") == "pix" else None,
        "desconto_pix_pct": best_pix.get("discount") if best_pix.get("paymentMethodId") == "pix" else None,
        "parcelado_melhor_n": best_inst.get("installment"),
        "parcelado_melhor_valor_parcela": best_inst.get("installmentAmount"),
        "parcelado_melhor_total": best_inst.get("totalAmount"),
        "parcelado_melhor_obs": best_inst.get("paymentMethodDescription"),
    })
    for pm in offer.get("paymentMethods") or []:
        plans = pm.get("installmentPlans") or []
        if not plans:
            continue
        pid = slug(pm["id"])
        free = [p for p in plans if not p.get("interestAmount")]
        last = max(plans, key=lambda p: p["installment"])
        row[f"pag_{pid}_max_parcelas_sem_juros"] = max((p["installment"] for p in free), default=None)
        row[f"pag_{pid}_max_parcelas"] = last["installment"]
        row[f"pag_{pid}_valor_parcela_max"] = last["installmentAmount"]
        row[f"pag_{pid}_total_max_parcelas"] = last["totalAmount"]

    # --- nota
    row["nota"] = (item.get("rating") or {}).get("score")
    row["n_avaliacoes"] = (item.get("rating") or {}).get("count")

    # --- cor / variações
    for att in item.get("attributes") or []:
        if att.get("type") == "color" or att.get("label") == "Cor":
            row["cor"] = att.get("current")
            row["variacoes_cor"] = " | ".join(v["value"] for v in att.get("values", []))
        else:
            row[f"variacao_{slug(att.get('label', 'x'))}"] = att.get("current")

    # --- embalagem (item.dimensions, em metros/kg)
    dim = item.get("dimensions") or {}
    row.update({"embalagem_largura_m": dim.get("width"), "embalagem_profundidade_m": dim.get("depth"),
                "embalagem_altura_m": dim.get("height"), "embalagem_peso_kg": dim.get("weight")})

    # --- ficha técnica -> uma coluna por chave
    for sec in item.get("factsheet") or []:
        for el in sec.get("elements") or []:
            key = el.get("keyName")
            vals = [str(v.get("value")) for v in el.get("elements", []) if v.get("value") not in (None, "")]
            if key and vals:
                row[f"ficha_{slug(key)}"] = " | ".join(vals)

    # --- normalização: peso, espuma e medidas (regex sobre a ficha técnica)
    ficha = {}
    for sec in item.get("factsheet") or []:
        for el in sec.get("elements") or []:
            if el.get("keyName"):
                ficha[el["keyName"]] = " | ".join(str(v.get("value")) for v in el.get("elements", []) if v.get("value") is not None)

    def fget(regex):
        for k, v in ficha.items():
            if re.fullmatch(regex, k.strip(), flags=re.I) and v:
                return v

    desc_txt = strip_html(item.get("description"))
    seats = seats_from(fget(r"quantidade (total )?de lugares"), item.get("title"))
    context = desc_txt + " | " + " | ".join(f"{k}: {v}" for k, v in ficha.items())

    # peso do produto (ficha) -> fallback: descrição. Peso suportado e peso com embalagem ficam separados.
    peso, peso_emb = weight_from_ficha(ficha, seats)
    row["peso_produto_kg"] = peso or weight_from_text(desc_txt)
    row["peso_fonte"] = "ficha" if peso else ("descricao" if row["peso_produto_kg"] else None)
    row["peso_com_embalagem_kg"] = peso_emb
    row["peso_suportado_txt"] = fget(r"peso (m[aá]ximo |total |real )?suportado( \(kg\))?") or None

    row["espuma_assento"] = parse_foam(fget(r"densidade (do assento|da espuma)"), fget(r"caracter[ií]sticas do assento"))
    row["espuma_codigos_descricao"] = parse_foam(desc_txt)

    # medidas: 1) chave "Largura x Profundidade x Altura" (ordem dada pelo nome)  2) "Dimensões do Produto"
    #          3) dimensões citadas na descrição  4) largura citada no título (ex.: 2,30m)
    dim_txt = fget(r"dimens[õo]es do produto")
    k_lpa = fget(r"largura x profundidade x altura")
    if k_lpa and len(re.findall(NUM, k_lpa)) >= 3:
        n = re.findall(NUM, k_lpa)
        d = dict(largura_cm=to_cm(n[0], "cm"), profundidade_cm=to_cm(n[1], "cm"), altura_cm=to_cm(n[2], "cm"),
                 profundidade_aberto_cm=None, altura_aberto_cm=None, dim_modo="chave_LxPxA", dim_complexa=False,
                 dim_ordem_incerta=False)
        dim_txt = dim_txt or k_lpa
    else:
        d = parse_dimensions(dim_txt, seats, context)
        if not d["largura_cm"]:
            m = re.search(r"dimens[õo]es?\s*:?\s*([^\n]{10,200})", desc_txt, re.I)
            if m:
                d = parse_dimensions(m.group(1), seats, context)
                dim_txt = dim_txt or m.group(1).strip()
                if d["largura_cm"]:
                    d["dim_modo"] = f"descricao:{d['dim_modo']}"
        if not d["largura_cm"]:
            w = width_from_title(item.get("title"))
            if w:
                d["largura_cm"], d["dim_modo"] = w, "titulo"
    row.update(d)
    row["dimensoes_txt"] = dim_txt
    row["dim_precisa_revisao"] = bool(
        d["dim_complexa"] or d["dim_ordem_incerta"] or not (d["largura_cm"] and d["profundidade_cm"] and d["altura_cm"])
    )

    row["destaques"] = " | ".join((data.get("keySellingPoint") or {}).get("items", []))
    row["descricao"] = strip_html(item.get("description"))
    row["n_imagens"] = len((item.get("media") or {}).get("images", []))
    row["tem_video"] = bool((item.get("media") or {}).get("videos"))
    return row


# ------------------------------------------------------------------ execução
def run():
    links = pd.read_csv(LINKS_CSV)["url"].dropna().tolist()
    if LIMIT:
        links = links[:LIMIT]

    done = set()
    if os.path.exists(RAW_JSONL):
        with open(RAW_JSONL, encoding="utf-8") as f:
            done = {json.loads(l)["link"] for l in f}
    todo = [l for l in links if l not in done]
    print(f"{len(done)} já baixados | {len(todo)} a baixar")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex, open(RAW_JSONL, "a", encoding="utf-8") as out:
        futs = [ex.submit(fetch_next_data, l) for l in todo]
        for i, fut in enumerate(as_completed(futs), 1):
            res = fut.result()
            if not res.get("error"):
                out.write(json.dumps(res, ensure_ascii=False) + "\n")
                out.flush()
            print(f"  [{i}/{len(todo)}] {'ok' if not res.get('error') else 'ERRO'}")

    # parsing 100% local: pode rodar de novo quantas vezes quiser sem custo
    rows = []
    with open(RAW_JSONL, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            if rec["link"] not in links:
                continue
            try:
                rows.append(parse_product(rec["link"], rec["next_data"]))
            except Exception as e:
                print(f"  falha ao parsear {rec['link'][:90]}: {e}")
    df = pd.DataFrame(rows)
    df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"{len(df)} produtos | {df.shape[1]} colunas -> {OUT_CSV}")
    return df


if __name__ == "__main__" or "get_ipython" in globals():
    df = run()

0 já baixados | 25 a baixar
  [1/25] ok
  [2/25] ok
  [3/25] ok
  [4/25] ok
  [5/25] ok
  [6/25] ok
  [7/25] ok
  [8/25] ok
  [9/25] ok
  [10/25] ok
  [11/25] ok
  [12/25] ok
  [13/25] ok
  [14/25] ok
  [15/25] ok
  [16/25] ok
  [17/25] ok
  [18/25] ok
  [19/25] ok
  [20/25] ok
  [21/25] ok
  [22/25] ok
  [23/25] ok
  [24/25] ok
  [25/25] ok
25 produtos | 197 colunas -> magalu_sofas_detalhes_local.csv
